# CNN-IDS para CAN Bus — Adaptação para Comma2K19

Adapta a metodologia do artigo AVTP (Elsevier 2021) para dados CAN bus reais coletados pelo dataset Comma2K19.

## Visão Geral
- **Protocolo**: CAN bus (Controller Area Network)
- **Dataset**: Comma2K19 — logs de condução real com `processed_log/CAN/raw_can/`
- **Ataque modelado**: Replay attack simulado — blocos de mensagens passadas são repetidos ciclicamente sobre dados reais
- **Campos usados**: `address` (CAN ID, 2 bytes: ID_hi + ID_lo) + `data` (8 bytes payload) = 10 bytes/mensagem
- **Pipeline**: raw_can → byte matrix (N, 10) → diff temporal mod 256 → nibbles → janela W=44 → CNN 2D
- **Input CNN**: `(44, 20, 1)` — 44 msgs x 20 nibbles (10 bytes x 2)
- **Filtragem**: Apenas barramento `src=0` (62% das mensagens — mantém sequência temporal correta)
- **Validação**: StratifiedKFold 5 folds
- **Resultados**: Accuracy ~89.5%, F1 ~94.1%, ROC-AUC ~91.2%

## Dataset no Kaggle
- `tkm2261/comma2k19-ld` — logs de condução Comma2K19

## Diferenças em relação ao artigo original
| | Artigo (AVTP) | Esta adaptação (CAN) |
|---|---|---|
| Bytes úteis | 58 bytes | 10 bytes |
| Nibbles/linha | 116 | 20 |
| Input CNN | (44, 116, 1) | **(44, 20, 1)** |
| Labels | Injeção real (pcap) | Replay attack simulado |


# CNN-IDS para CAN Bus — Adaptação do artigo AVTP para Comma2K19

Adapta a metodologia do artigo (Elsevier 2021) para dados CAN bus reais do Comma2K19.

| | Artigo (AVTP) | Esta adaptação (CAN bus) |
|---|---|---|
| **Dados** | Pcap AVTP, 438 bytes/pacote | raw_can: address + data (S8) |
| **Bytes úteis** | 58 bytes | 10 bytes: ID_hi, ID_lo, D0..D7 |
| **Nibbles/linha** | 116 | 20 |
| **Input CNN** | (44, 116, 1) | **(44, 20, 1)** |
| **Labels** | Injeção real (pcap) | Replay attack simulado sobre CAN real |

A pipeline de feature engineering é idêntica: diff temporal → mod 256 → nibbles → janela W=44.

## Estrutura do Comma2K19 usada
```
processed_log/CAN/raw_can/
  address   (N,)  int64    CAN ID
  data      (N,)  |S8      8 bytes de payload
  src       (N,)  int64    barramento
  t         (N,)  float64  timestamp
```


In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

2026-06-10 15:12:55.331566: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781104375.504927      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781104375.555759      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781104375.953445      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781104375.953485      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781104375.953488      23 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import time

def setup_gpu_optimization():
    """Ativa mixed precision, XLA e memory growth — executa 1x antes do loop."""
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.config.optimizer.set_jit(True)  # XLA
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU otimizada: Mixed Precision (float16) + XLA + Memory Growth')

setup_gpu_optimization()


GPU otimizada: Mixed Precision (float16) + XLA + Memory Growth


In [3]:
# Path correto do dataset no Kaggle
DATASET_PATH  = '/kaggle/input/datasets/tkm2261/comma2k19-ld'

WINDOW_SIZE   = 44
N_BYTES       = 10
N_COLS        = N_BYTES * 2   # 20 nibbles

BATCH_SIZE    = 64
EPOCHS        = 30
LEARNING_RATE = 0.001
PATIENCE_ES   = 5

BLOCK_SIZE    = 36
ATTACK_PROB   = 0.25
MAX_SEGMENTS  = 20    # 20 segs x ~79 MB (src=0, 62%) = ~1.6 GB de X
MIN_MSGS      = WINDOW_SIZE + 200
SEED          = 42
SRC_FILTER    = 0     # barramento principal (62% das msgs)

# RAM estimada: 144k msgs x 62% = ~89k msgs/seg
# 89k janelas x 44 x 20 bytes = ~78 MB/seg
# 20 segmentos = ~1.6 GB — bem dentro dos 20 GB disponiveis
print(f'Input CNN: ({WINDOW_SIZE}, {N_COLS}, 1)')
print(f'RAM estimada para X (20 segs, src=0): ~{20 * 78} MB')


Input CNN: (44, 20, 1)
RAM estimada para X (20 segs, src=0): ~1560 MB


## 2. Leitura dos dados CAN

O campo `data` tem dtype `|S8` (string de bytes de tamanho fixo 8).  
`tobytes()` serializa todos em sequência; `reshape(N, 8)` divide em linhas — cada linha = 1 mensagem CAN.


In [4]:
def raw_can_para_byte_matrix(raw_can_path, src_filter=0):
    """
    Le os arquivos raw_can do Comma2K19.
    Retorna matriz (N, 10) uint8: [ID_hi, ID_lo, D0..D7]

    src_filter=0: barramento principal (62% das msgs).
    Os 4 barramentos estao intercalados no tempo — mistura-los
    corrompe a sequencia temporal que o modelo precisa aprender.
    """
    address = np.load(os.path.join(raw_can_path, 'address'), allow_pickle=True)
    data_s8 = np.load(os.path.join(raw_can_path, 'data'),    allow_pickle=True)
    src     = np.load(os.path.join(raw_can_path, 'src'),     allow_pickle=True)

    mask    = src == src_filter
    address = address[mask]
    data_s8 = data_s8[mask]

    N = len(address)
    if N == 0:
        return np.empty((0, 10), dtype=np.uint8)

    # |S8 -> (N, 8) uint8
    data_bytes = np.frombuffer(data_s8.tobytes(), dtype=np.uint8).reshape(N, 8)

    matrix = np.zeros((N, 10), dtype=np.uint8)
    matrix[:, 0] = (address >> 8) & 0xFF
    matrix[:, 1] =  address       & 0xFF
    matrix[:, 2:] = data_bytes
    return matrix


def encontrar_segmentos_can(dataset_path, max_segments):
    """Busca pastas raw_can com address, data, src e t."""
    paths = []
    for root, dirs, files in os.walk(dataset_path):
        if os.path.basename(root) == 'raw_can':
            if all(os.path.exists(os.path.join(root, f)) for f in ['address', 'data', 'src', 't']):
                paths.append(root)
                if len(paths) >= max_segments:
                    break
    return paths


segmentos = encontrar_segmentos_can(DATASET_PATH, MAX_SEGMENTS)
print(f'Segmentos raw_can encontrados: {len(segmentos)}')
for p in segmentos[:5]:
    print(' ', p)
if len(segmentos) > 5:
    print(f'  ... e mais {len(segmentos) - 5}')
if len(segmentos) == 0:
    raise RuntimeError(f'Nenhuma pasta raw_can encontrada em: {DATASET_PATH}')


Segmentos raw_can encontrados: 20
  /kaggle/input/datasets/tkm2261/comma2k19-ld/scb89/processed_log/CAN/raw_can
  /kaggle/input/datasets/tkm2261/comma2k19-ld/scb72/processed_log/CAN/raw_can
  /kaggle/input/datasets/tkm2261/comma2k19-ld/scb68/processed_log/CAN/raw_can
  /kaggle/input/datasets/tkm2261/comma2k19-ld/scb65/processed_log/CAN/raw_can
  /kaggle/input/datasets/tkm2261/comma2k19-ld/scb83/processed_log/CAN/raw_can
  ... e mais 15


## Diagnóstico — inspecionar primeiro segmento
Rode esta célula para confirmar que a leitura está correta antes de continuar.


In [5]:
if segmentos:
    p = segmentos[0]
    print('Path:', p)

    # Distribuição de barramentos
    src = np.load(os.path.join(p, 'src'), allow_pickle=True)
    unique, counts = np.unique(src, return_counts=True)
    print('Distribuicao de barramentos (src):')
    for u, c in zip(unique, counts):
        print(f'  src={u}: {c:,} msgs ({100*c/len(src):.1f}%)')

    # Shapes brutos
    print()
    for f in ['address', 'data', 't', 'src']:
        fp = os.path.join(p, f)
        if os.path.exists(fp):
            arr = np.load(fp, allow_pickle=True)
            print(f'  {f:10s} shape={arr.shape}  dtype={arr.dtype}')

    # Teste de leitura filtrada (src=0)
    print()
    mat = raw_can_para_byte_matrix(p, src_filter=0)
    print(f'byte_matrix shape apos filtro src=0: {mat.shape}')
    print(f'primeiras 2 linhas (ID_hi, ID_lo, D0..D7):')
    print(mat[:2])

Path: /kaggle/input/datasets/tkm2261/comma2k19-ld/scb89/processed_log/CAN/raw_can
Distribuicao de barramentos (src):
  src=0: 89,413 msgs (62.1%)
  src=1: 40,800 msgs (28.3%)
  src=128: 11,400 msgs (7.9%)
  src=129: 2,400 msgs (1.7%)

  address    shape=(144013,)  dtype=int64
  data       shape=(144013,)  dtype=|S8
  t          shape=(144013,)  dtype=float64
  src        shape=(144013,)  dtype=int64

byte_matrix shape apos filtro src=0: (89413, 10)
primeiras 2 linhas (ID_hi, ID_lo, D0..D7):
[[  0 148 128 164 135 233 254   0  12 127]
 [  2  33  36   0   0   0   0  13   0   0]]


## 3. Feature engineering — idêntico ao artigo

1. **Diff temporal**: `delta_i = (x_i - x_{i-1}) mod 256`
2. **Split em nibbles**: byte → nibble alto (bits 7-4) + nibble baixo (bits 3-0)
3. **Janela deslizante W=44**, stride=1


In [6]:
def calcular_diff_nibbles(byte_matrix):
    """
    Diff temporal mod 256 + split em nibbles.
    Entrada:  (N, 10) uint8
    Saida:    (N-1, 20) uint8
    """
    diff = np.diff(byte_matrix.astype(np.int16), axis=0)
    diff_mod = np.mod(diff, 256).astype(np.uint8)

    high = (diff_mod >> 4) & 0x0F
    low  =  diff_mod       & 0x0F
    nibbles = np.stack([high, low], axis=-1).reshape(len(diff_mod), -1)
    return nibbles.astype(np.uint8)


def janela_deslizante(x_data, y_data, window_size=44):
    """
    Janela deslizante stride=1.
    Label = 1 se qualquer mensagem na janela for ataque.
    """
    N = len(x_data)
    n_jan = N - window_size + 1
    if n_jan <= 0:
        return np.empty((0, window_size, x_data.shape[1]), dtype=np.uint8), np.empty(0, dtype=np.uint8)

    idx = np.arange(window_size)[None, :] + np.arange(n_jan)[:, None]
    X_win = x_data[idx]
    y_win = y_data[idx].max(axis=1)
    return X_win.astype(np.uint8), y_win.astype(np.uint8)

print('Feature engineering pronto.')

Feature engineering pronto.


## 4. Simulação de replay attack

Mesmo mecanismo do artigo: um bloco de mensagens passadas é repetido ciclicamente,
substituindo mensagens legítimas. Aqui aplicado sobre os dados CAN reais do Comma2K19.


In [7]:
def simular_replay_attack(byte_matrix, block_size=36, attack_prob=0.25, seed=42):
    """
    Injeta ataques de replay sobre dados CAN reais.
    Retorna: (injected_matrix, labels) ambos shape (N,)
    """
    rng = np.random.default_rng(seed)
    N = len(byte_matrix)
    injected = byte_matrix.copy()
    labels = np.zeros(N, dtype=np.uint8)

    idx = WINDOW_SIZE + block_size
    while idx < N - 150:
        if rng.random() < attack_prob:
            atk_len = int(rng.integers(50, 111))
            end = min(idx + atk_len, N)
            buffer = injected[idx - block_size : idx].copy()
            for j in range(idx, end):
                injected[j] = buffer[(j - idx) % block_size]
                labels[j] = 1
            idx = end + 60
        else:
            idx += 1

    return injected, labels

print('Funcao de ataque pronta.')

Funcao de ataque pronta.


## 5. Carregamento e preprocessamento de todos os segmentos


In [8]:
X_all = []
y_all = []
total_benigno = 0
total_ataque  = 0

for i, raw_can_path in enumerate(segmentos):
    try:
        byte_mat = raw_can_para_byte_matrix(raw_can_path, src_filter=SRC_FILTER)

        if len(byte_mat) < MIN_MSGS:
            print(f'  [!] Seg {i+1}: poucas msgs ({len(byte_mat)}), pulando.')
            continue

        injected, labels = simular_replay_attack(
            byte_mat, block_size=BLOCK_SIZE, attack_prob=ATTACK_PROB, seed=SEED + i
        )
        del byte_mat

        nibbles = calcular_diff_nibbles(injected)
        labels_diff = labels[1:]
        del injected, labels

        X_seg, y_seg = janela_deslizante(nibbles, labels_diff, window_size=WINDOW_SIZE)
        del nibbles, labels_diff

        if len(X_seg) == 0:
            continue

        X_all.append(X_seg)
        y_all.append(y_seg)

        n_ben = int((y_seg == 0).sum())
        n_atk = int((y_seg == 1).sum())
        total_benigno += n_ben
        total_ataque  += n_atk

        ram_gb = sum(x.nbytes for x in X_all) / 1e9
        print(f'  [+] Seg {i+1:2d}: {len(X_seg):6d} janelas '
              f'| benigno={n_ben} ataque={n_atk} '
              f'| RAM acumulada: {ram_gb:.2f} GB')
        gc.collect()

    except Exception as e:
        print(f'  [!] Seg {i+1} erro: {e}')

if len(X_all) == 0:
    raise RuntimeError('Nenhum segmento valido carregado.')

print('\nConcatenando...')
X = np.concatenate(X_all, axis=0)[..., np.newaxis]  # (N, 44, 20, 1)
y = np.concatenate(y_all, axis=0)
del X_all, y_all
gc.collect()

print(f'X shape : {X.shape}  ({X.nbytes/1e9:.2f} GB)')
print(f'y shape : {y.shape}')
print(f'Benigno : {total_benigno:,}  ({100*total_benigno/(total_benigno+total_ataque):.1f}%)')
print(f'Ataque  : {total_ataque:,}  ({100*total_ataque/(total_benigno+total_ataque):.1f}%)')


  [+] Seg  1:  89369 janelas | benigno=12655 ataque=76714 | RAM acumulada: 0.08 GB
  [+] Seg  2:  89361 janelas | benigno=12555 ataque=76806 | RAM acumulada: 0.16 GB
  [+] Seg  3:  89365 janelas | benigno=12454 ataque=76911 | RAM acumulada: 0.24 GB
  [+] Seg  4:  53803 janelas | benigno=7565 ataque=46238 | RAM acumulada: 0.28 GB
  [+] Seg  5:  89363 janelas | benigno=12357 ataque=77006 | RAM acumulada: 0.36 GB
  [+] Seg  6:  89369 janelas | benigno=12722 ataque=76647 | RAM acumulada: 0.44 GB
  [+] Seg  7:  89360 janelas | benigno=12529 ataque=76831 | RAM acumulada: 0.52 GB
  [+] Seg  8:  89371 janelas | benigno=12538 ataque=76833 | RAM acumulada: 0.60 GB
  [+] Seg  9:  89361 janelas | benigno=12525 ataque=76836 | RAM acumulada: 0.68 GB
  [+] Seg 10:  89364 janelas | benigno=12475 ataque=76889 | RAM acumulada: 0.76 GB
  [+] Seg 11:  53766 janelas | benigno=7511 ataque=46255 | RAM acumulada: 0.80 GB
  [+] Seg 12:  89366 janelas | benigno=12589 ataque=76777 | RAM acumulada: 0.88 GB
  [+] 

## 6. Arquitetura CNN 2D — Repositório oficial dos autores

Modelo baseado no código publicado no GitHub do artigo (`create_optimized_cnn`).
Adaptações para CAN bus:
- Input: **(44, 20, 1)** em vez de (44, 116, 1)
- `dtype='float32'` na saída mantido — crítico para estabilidade com mixed precision


In [9]:
def criar_modelo_cnn(input_shape=(WINDOW_SIZE, N_COLS, 1)):
    """
    Arquitetura do repositório oficial dos autores (create_optimized_cnn).
    Adaptada para CAN bus: input (44, 20, 1) em vez de (44, 116, 1).

    Diferenças em relação à Tabela 2 do paper:
      - Kernel (3,3) em vez de (5,5)
      - 3 blocos conv em vez de 2 (adiciona Conv2D 128)
      - GlobalAveragePooling2D em vez de Flatten
      - Dropout entre blocos (0.2, 0.3, 0.4) em vez de L2
      - dtype='float32' na saída — essencial com mixed_float16
    """
    inputs = tf.keras.Input(shape=input_shape)

    # Bloco 1
    x = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Bloco 2
    x = tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # Bloco 3
    x = tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    # Classificador
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    # CRITICO: float32 explicito na saida para estabilidade com mixed_float16
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs, name='CNN_IDS_CAN')
    return model


# Verificar arquitetura
modelo_tmp = criar_modelo_cnn()
modelo_tmp.summary()
del modelo_tmp
tf.keras.backend.clear_session()


I0000 00:00:1781104401.274096      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "CNN_IDS_CAN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 44, 20, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 44, 20, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 44, 20, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 22, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 10, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 22, 10, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 5, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 11, 5, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 11, 5, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,377 (396.00 KB)

 Trainable params: 101,185 (395.25 KB)

 Non-trainable params: 192 (768.00 B)

## 7. Validação cruzada 5-Fold

Mesma estratégia do artigo: StratifiedKFold, 5 splits, random_state=1.


In [10]:
def build_fast_dataset(X, y, batch_size, shuffle=True):
    """
    Pipeline tf.data otimizado:
    - X fica em uint8 na RAM; cast float32 + norm ocorre por batch na GPU
    - cache() mantém batches processados na RAM entre epochs
    - prefetch solapa CPU/GPU
    """
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, len(X)), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    def process_batch(x, y):
        x = tf.cast(x, tf.float32) / 15.0
        return x, tf.cast(y, tf.int32)
    ds = ds.map(process_batch, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache()        # cacheia batches na RAM após 1a epoch
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def resetar_seeds(seed=SEED):
    np.random.seed(seed)
    random.seed(seed + 1)
    tf.random.set_seed(seed + 2)


OUTPUT_PATH = '/kaggle/working'
BATCH_SIZE_TRAIN = 256  # maior batch = treino mais rapido na P100
                        # LR e escalado proporcionalmente

skf = StratifiedKFold(n_splits=5, random_state=1, shuffle=True)
metricas    = []
fold        = 1

# Acumuladores para plots e matriz de confusão consolidada
all_histories = []   # {'fold', 'loss', 'val_loss'} de cada fold
all_y_true    = []   # labels reais acumulados
all_y_pred    = []   # predições binárias acumuladas
all_y_prob    = []   # probabilidades acumuladas

for train_idx, val_idx in skf.split(X, y):
    print(f'\n{"="*55}')
    print(f' Fold {fold}/5')
    print(f'{"="*55}')

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    print(f'  Treino: {len(X_train):,}  Validacao: {len(X_val):,}')
    print(f'  X_train RAM : {X_train.nbytes/1e9:.2f} GB (uint8)')
    print(f'  Equiv float32 sem tf.data: {X_train.nbytes*4/1e9:.2f} GB')

    train_ds = build_fast_dataset(X_train, y_train, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_ds   = build_fast_dataset(X_val,   y_val,   batch_size=BATCH_SIZE_TRAIN, shuffle=False)

    resetar_seeds()

    # Modelo — saida float32 explícita para estabilidade com mixed precision
    model = criar_modelo_cnn(input_shape=(WINDOW_SIZE, N_COLS, 1))

    # LR escalado com batch size (linear scaling rule)
    lr_scaled = LEARNING_RATE * (BATCH_SIZE_TRAIN / 64)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_scaled),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
        jit_compile=True  # XLA
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE_ES,
            restore_best_weights=True, mode='min', verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            f'{OUTPUT_PATH}/model_fold{fold}.h5',
            monitor='val_accuracy', save_best_only=True, mode='max', verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3,
            min_lr=1e-6, verbose=1
        )
    ]

    t0 = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=2
    )
    elapsed = time.time() - t0
    print(f'  Fold {fold} concluido em {elapsed:.1f}s | Epocas: {len(history.history["loss"])}')

    # Salvar history para gráfico de loss
    all_histories.append({
        'fold':     fold,
        'loss':     history.history['loss'],
        'val_loss': history.history.get('val_loss', [])
    })

    prob_pred = model.predict(val_ds, verbose=0).flatten()
    y_pred    = (prob_pred >= 0.5).astype(int)

    # Acumular para matriz de confusão consolidada
    all_y_true.extend(y_val.tolist())
    all_y_pred.extend(y_pred.tolist())
    all_y_prob.extend(prob_pred.tolist())

    acc  = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec  = recall_score(y_val, y_pred, zero_division=0)
    f1   = f1_score(y_val, y_pred, zero_division=0)
    auc  = roc_auc_score(y_val, prob_pred)

    metricas.append({'fold': fold, 'acc': acc, 'prec': prec,
                     'recall': rec, 'f1': f1, 'auc': auc})

    # Salvar CSV incremental (útil se o kernel for interrompido)
    pd.DataFrame(metricas).to_csv(f'{OUTPUT_PATH}/metricas_por_fold.csv', index=False)
    print(f'  Acc={acc:.4f}  Prec={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}')

    del model, history, train_ds, val_ds, prob_pred, y_pred, X_train, X_val, y_train, y_val
    gc.collect()
    tf.keras.backend.clear_session()
    fold += 1

print('\nTreinamento completo.')



 Fold 1/5
  Treino: 1,344,441  Validacao: 336,111
  X_train RAM : 1.18 GB (uint8)
  Equiv float32 sem tf.data: 4.73 GB
Epoch 1/30


I0000 00:00:1781104409.992346      68 service.cc:152] XLA service 0x7abd94003890 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781104409.992388      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781104410.021245      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1781104410.164286      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


21007/21007 - 103s - 5ms/step - accuracy: 0.8601 - auc: 0.7013 - loss: 0.3718 - val_accuracy: 0.8620 - val_auc: 0.7442 - val_loss: 0.3615 - learning_rate: 0.0040
Epoch 2/30


21007/21007 - 86s - 4ms/step - accuracy: 0.8630 - auc: 0.7717 - loss: 0.3432 - val_accuracy: 0.8710 - val_auc: 0.8071 - val_loss: 0.3234 - learning_rate: 0.0040
Epoch 3/30


21007/21007 - 86s - 4ms/step - accuracy: 0.8665 - auc: 0.8009 - loss: 0.3284 - val_accuracy: 0.8744 - val_auc: 0.8392 - val_loss: 0.3045 - learning_rate: 0.0040
Epoch 4/30


...

21007/21007 - 90s - 4ms/step - accuracy: 0.8828 - auc: 0.8751 - loss: 0.2778 - val_accuracy: 0.8943 - val_auc: 0.9115 - val_loss: 0.2449 - learning_rate: 0.0020
Epoch 21/30
21007/21007 - 90s - 4ms/step - accuracy: 0.8831 - auc: 0.8769 - loss: 0.2764 - val_accuracy: 0.8936 - val_auc: 0.9123 - val_loss: 0.2448 - learning_rate: 0.0020
Epoch 22/30


21007/21007 - 90s - 4ms/step - accuracy: 0.8838 - auc: 0.8784 - loss: 0.2750 - val_accuracy: 0.8948 - val_auc: 0.9146 - val_loss: 0.2420 - learning_rate: 0.0020
Epoch 23/30


21007/21007 - 90s - 4ms/step - accuracy: 0.8839 - auc: 0.8789 - loss: 0.2744 - val_accuracy: 0.8961 - val_auc: 0.9149 - val_loss: 0.2418 - learning_rate: 0.0020
Epoch 24/30
21007/21007 - 91s - 4ms/step - accuracy: 0.8839 - auc: 0.8795 - loss: 0.2740 - val_accuracy: 0.8949 - val_auc: 0.9160 - val_loss: 0.2419 - learning_rate: 0.0020
Epoch 25/30
21007/21007 - 90s - 4ms/step - accuracy: 0.8840 - auc: 0.8796 - loss: 0.2740 - val_accuracy: 0.8951 - val_auc: 0.9149 - val_loss: 0.2423 - learning_rate: 0.0020
Epoch 26/30


21007/21007 - 90s - 4ms/step - accuracy: 0.8841 - auc: 0.8801 - loss: 0.2735 - val_accuracy: 0.8970 - val_auc: 0.9158 - val_loss: 0.2392 - learning_rate: 0.0020
Epoch 27/30
21007/21007 - 91s - 4ms/step - accuracy: 0.8845 - auc: 0.8809 - loss: 0.2728 - val_accuracy: 0.8955 - val_auc: 0.9161 - val_loss: 0.2395 - learning_rate: 0.0020
Epoch 28/30
21007/21007 - 90s - 4ms/step - accuracy: 0.8847 - auc: 0.8818 - loss: 0.2719 - val_accuracy: 0.8965 - val_auc: 0.9164 - val_loss: 0.2419 - learning_rate: 0.0020
Epoch 29/30

Epoch 29: ReduceLROnPlateau reducing learning rate to 0.0010000000474974513.
21007/21007 - 90s - 4ms/step - accuracy: 0.8847 - auc: 0.8824 - loss: 0.2716 - val_accuracy: 0.8954 - val_auc: 0.9149 - val_loss: 0.2408 - learning_rate: 0.0020
Epoch 30/30


21007/21007 - 94s - 4ms/step - accuracy: 0.8855 - auc: 0.8848 - loss: 0.2692 - val_accuracy: 0.8986 - val_auc: 0.9205 - val_loss: 0.2351 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 30.
  Fold 5 concluido em 2736.3s | Epocas: 30
  Acc=0.8986  Prec=0.9208  Recall=0.9650  F1=0.9424  AUC=0.9206

Treinamento completo.


## 8. Resultados consolidados


In [11]:
import pandas as pd

# CSV por fold
df = pd.DataFrame(metricas)
df.to_csv(f'{OUTPUT_PATH}/metricas_por_fold.csv', index=False)

# CSV com médias e desvio padrão
mean_row = df[['acc','prec','recall','f1','auc']].mean()
std_row  = df[['acc','prec','recall','f1','auc']].std()
summary  = pd.concat([
    df,
    pd.DataFrame([['mean'] + mean_row.tolist()], columns=df.columns),
    pd.DataFrame([['std']  + std_row.tolist()],  columns=df.columns),
], ignore_index=True)
summary.to_csv(f'{OUTPUT_PATH}/metricas_resumo.csv', index=False)

print('Metricas por fold:')
print(df.to_string(index=False, float_format='{:.4f}'.format))
print('\nMedias (+- desvio padrao):')
for col in ['acc', 'prec', 'recall', 'f1', 'auc']:
    print(f'  {col:8s}: {df[col].mean():.4f} +- {df[col].std():.4f}')
print(f'\nCSV salvo em {OUTPUT_PATH}/')


Metricas por fold:
 fold    acc   prec  recall     f1    auc
    1 0.8896 0.9058  0.9726 0.9380 0.8995
    2 0.8941 0.9225  0.9573 0.9395 0.9040
    3 0.8953 0.9233  0.9578 0.9402 0.9133
    4 0.9003 0.9284  0.9579 0.9429 0.9222
    5 0.8986 0.9208  0.9650 0.9424 0.9206

Medias (+- desvio padrao):
  acc     : 0.8956 +- 0.0042
  prec    : 0.9202 +- 0.0085
  recall  : 0.9621 +- 0.0067
  f1      : 0.9406 +- 0.0020
  auc     : 0.9119 +- 0.0100

CSV salvo em /kaggle/working/


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

n_folds = len(all_histories)
cols = min(n_folds, 3)
rows = (n_folds + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
axes_flat = np.array(axes).flatten() if n_folds > 1 else [axes]

for i, h in enumerate(all_histories):
    ax = axes_flat[i]
    ax.plot(h['loss'],     label='Train Loss', color='#1F3864', linewidth=2)
    if h['val_loss']:
        ax.plot(h['val_loss'], label='Val Loss', color='#E05A2B', linewidth=2, linestyle='--')
    ax.set_title(f'Fold {h["fold"]}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Curva de Loss — Treino vs Validacao por Fold', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: loss_curves.png')


Salvo: loss_curves.png


In [13]:
metric_cols   = ['acc', 'prec', 'recall', 'f1', 'auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#1F3864', '#2E75B6', '#70AD47', '#E05A2B', '#7030A0']

fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(df))
width = 0.15

for j, (col, label, color) in enumerate(zip(metric_cols, metric_labels, colors)):
    ax.bar(x + j*width, df[col], width, label=label,
           color=color, alpha=0.85, edgecolor='white')
    # linha de média
    ax.hlines(df[col].mean(),
              x[0]+j*width - width/2, x[-1]+j*width + width/2,
              colors=color, linestyles='dashed', linewidth=1.2, alpha=0.6)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Metricas por Fold — CNN-IDS CAN Bus (Comma2K19)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels([f'Fold {i+1}' for i in range(len(df))])
ax.set_ylim(0, 1.08)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/metricas_por_fold.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: metricas_por_fold.png')


Salvo: metricas_por_fold.png


In [14]:
from sklearn.metrics import confusion_matrix, classification_report

y_true_all = np.array(all_y_true)
y_pred_all = np.array(all_y_pred)

cm      = confusion_matrix(y_true_all, y_pred_all)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Contagens absolutas', 'Normalizada por classe real (%)'],
    ['d', '.2%']
):
    im = ax.imshow(data, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Predito', fontsize=12)
    ax.set_ylabel('Real', fontsize=12)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Benigno', 'Ataque'], fontsize=11)
    ax.set_yticklabels(['Benigno', 'Ataque'], fontsize=11)
    thresh = data.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(data[i, j], fmt),
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if data[i, j] > thresh else '#1F3864')

fig.suptitle('Matriz de Confusao Consolidada — Todos os Folds', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'FPR (taxa falso positivo): {fp/(fp+tn):.4f}')
print(f'FNR (taxa falso negativo): {fn/(fn+tp):.4f}')
print()
print(classification_report(y_true_all, y_pred_all, target_names=['Benigno', 'Ataque']))
print('Salvo: confusion_matrix.png')


TN=115,218  FP=120,775  FN=54,719  TP=1,389,840
FPR (taxa falso positivo): 0.5118
FNR (taxa falso negativo): 0.0379

              precision    recall  f1-score   support

     Benigno       0.68      0.49      0.57    235993
      Ataque       0.92      0.96      0.94   1444559

    accuracy                           0.90   1680552
   macro avg       0.80      0.73      0.75   1680552
weighted avg       0.89      0.90      0.89   1680552

Salvo: confusion_matrix.png


In [15]:
import os

outputs = [
    'metricas_por_fold.csv',
    'metricas_resumo.csv',
    'loss_curves.png',
    'metricas_por_fold.png',
    'confusion_matrix.png',
]

print('Arquivos gerados em /kaggle/working/:')
print('-' * 48)
for fname in outputs:
    path = f'{OUTPUT_PATH}/{fname}'
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  OK    {fname:35s} {size_kb:.1f} KB')
    else:
        print(f'  FALTA {fname}')


Arquivos gerados em /kaggle/working/:
------------------------------------------------
  OK    metricas_por_fold.csv               0.5 KB
  OK    metricas_resumo.csv                 0.7 KB
  OK    loss_curves.png                     215.5 KB
  OK    metricas_por_fold.png               52.5 KB
  OK    confusion_matrix.png                79.4 KB
